In [ ]:
import control as con
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy

from feed_conc_ctrl.plot_utils import make_tsplots


## PI Regulator in Interactive Form

$$
G_c(s) = K_c \frac{T_i s + 1}{T_i s}
$$                                                                                
   
where:                                                                                                                                                        
 - $K_c$ : proportional gain
 - $T_i$ : integral time (reset time)
 - $s$ : Laplace variable


In time domain:

$$
u(t)=K_c \left( e(t)+\frac{1}{T_i} \int_0^t e(\tau) d \tau \right)
$$

In [ ]:
# PI Regulator in Interactive Form

s = sympy.Symbol("s")
K_c, T_i = sympy.symbols("K_c, T_i", positive=True)

G_c = K_c * (T_i * s + 1) / (T_i * s)
G_c

In [ ]:
# Time-domain form: u(t) = ?
# G_c(s) = U(s)/E(s), decompose: K_c + K_c/(T_i*s)
# Inverse Laplace of each term, then convolve with e(t):

t, tau = sympy.symbols("t, tau", positive=True)
e = sympy.Function("e")

u_t = K_c * e(t) + (K_c / T_i) * sympy.Integral(e(tau), (tau, 0, t))
sympy.Eq(sympy.Function("u")(t), u_t)

## PID Regulator

## Parallel Form (Åström and Hägglund)

$$
u(t) = K_c \left(e(t)+\frac{1}{T_i} \int_0^t e(\tau) d \tau+T_d \frac{d e(t)}{d t}\right)
$$

## Linear Parametrization Form

$$
u(t)=k_p e(t)+k_i \int_0^t e(\tau) d \tau+k_d \frac{d e(t)}{d t},
$$

## Interactive (Series) Form

Laplace domain:

$$
U(s) = G_c(s) E(s)
$$

$$
G_c(s) = K_c \frac{(T_i s + 1)(T_d s + 1)}{T_i s \,(T_f s + 1)}
$$

Time domain:

$$
T_f \dot{u}(t) + u(t) = K_c \left[ T_d \frac{d e(t)}{d t} + \left(1 + \frac{T_d}{T_i}\right) e(t) + \frac{1}{T_i} \int_0^t e(\tau)\, d\tau \right]
$$

where:
 - $K_c$ : proportional gain
 - $T_i$ : integral time constant (reset time)
 - $T_d$ : derivative time constant
 - $T_f$ : derivative filter time constant
 - $s$ : Laplace variable

In [ ]:
# PID Controller in Interactive Form

T_d, T_f = sympy.symbols("T_d, T_f", positive=True)

G_c_pid = K_c * (T_i * s + 1) * (T_d * s + 1) / (T_i * s * (T_f * s + 1))
G_c_pid

In [ ]:
# Time-domain form (implicit ODE for u(t)):
# Cross-multiply: U(s)*T_i*s*(T_f*s+1) = K_c*(T_i*s+1)*(T_d*s+1)*E(s)
# Divide by T_i*s: (T_f*s+1)*U(s) = K_c*[T_d*s + (1+T_d/T_i) + 1/(T_i*s)]*E(s)
# Inverse Laplace:

u = sympy.Function("u")

lhs = T_f * u(t).diff(t) + u(t)
rhs = K_c * (
    T_d * e(t).diff(t)
    + (1 + T_d / T_i) * e(t)
    + sympy.Integral(e(tau), (tau, 0, t)) / T_i
)
sympy.Eq(lhs, rhs)

### Converting Interactive Form Parameters to Parallel Form

Setting $T_f = 0$ and expanding:

$$
G_c(s) = K_c \frac{(T_i s + 1)(T_d s + 1)}{T_i s}
= K_c T_d \, s + K_c\!\left(1 + \frac{T_d}{T_i}\right) + \frac{K_c}{T_i s}
$$

Matching coefficients with $k_d s + k_p + k_i / s$:

$$
k_p = K_c\!\left(\frac{T_i + T_d}{T_i}\right), \qquad k_i = \frac{K_c}{T_i}, \qquad k_d = K_c T_d
$$

In [ ]:
# Derive parallel parameters by expanding G_c_pid with T_f=0
G_c_ideal = G_c_pid.subs(T_f, 0)
expanded = sympy.apart(G_c_ideal, s)  # partial fractions in s

k_p, k_i, k_d = sympy.symbols("k_p k_i k_d")


def pid_interactive_to_linear_parallel(K_c, T_i, T_d=0.0):
    """Convert interactive-form PID parameters to continuous-time linear parallel gains.

    Returns k_p, k_i, k_d where u(t) = k_p*e + k_i*integral(e) + k_d*de/dt.
    """
    k_p = K_c * (T_i + T_d) / T_i
    k_i = K_c / T_i
    k_d = K_c * T_d
    return k_p, k_i, k_d


def pid_interactive_ct_to_linear_parallel_dt(K_c, T_i, T_d=0.0, Ts=1.0):
    """Convert interactive-form CT PID parameters to discrete-time linear parallel gains
    compatible with the reference PIDController.

    In the reference PID incremental form:
        Dui = ki * e * Tx          (Tx = actual_dt / Ts)
        ud  = -kd * dyf            (dyf is normalized: dyf = Ts * d(yf)/dt)

    To match continuous-time integral/derivative actions:
        ki_dt = ki_ct * Ts         (scales with sample time)
        kd_dt = kd_ct / Ts         (inverse scaling)
        kp_dt = kp_ct              (unchanged — no time dimension)

    Args:
        K_c: Interactive-form proportional gain
        T_i: Integral time constant [s]
        T_d: Derivative time constant [s]
        Ts:  Sample time [s]

    Returns:
        k_p, k_i, k_d: Discrete-time parallel gains for PIDController
    """
    k_p = K_c * (T_i + T_d) / T_i
    k_i = K_c / T_i * Ts
    k_d = K_c * T_d / Ts
    return k_p, k_i, k_d


k_p, k_i, k_d = pid_interactive_to_linear_parallel(K_c, T_i, T_d)

for name, expr in [("k_p", k_p), ("k_i", k_i), ("k_d", k_d)]:
    display(sympy.Eq(sympy.Symbol(name), expr))

## Setpoint Filter

First order

$$
Y_f(s)=\frac{1}{s T_f+1} Y(s)
$$

Second order

$$
Y_f(s)=\frac{1}{\left(s T_f+1\right)^2} Y(s)
$$

## Test Case 1

In [ ]:
# First-order stable plant
K_p = 2.0
T_1 = 2.0

G_p = con.tf([K_p], [T_1, 1])
print(G_p)

In [ ]:
con.pzmap(G_p)
plt.show()

In [ ]:
# Time vector and step input starting at t=1
t = np.linspace(0, 20, 201)
u = np.where(t >= 1, 1.0, 0.0)

response = con.forced_response(G_p, T=t, U=u)
y = response.y[0]

data = pd.DataFrame(
    {"input": u, "output": y}, index=pd.Index(t, name="Time (t)")
)

plot_info = {
    "Step Response": {
        "Input $u(t)$": {
            "var_name": "input",
            "kind": "step",
            "linestyle": "--",
            "color": "k",
        },
        "Output $y(t)$": {"var_name": "output"},
    }
}

fig, axes = make_tsplots(data, plot_info=plot_info)
plt.tight_layout()
plt.show()

In [ ]:
# PI design rule (lambda tuning: closed-loop TC = T_1)
K_c = 1.0 / K_p
T_i = T_1

K_c, T_i

In [ ]:
G_c = con.tf([K_c * T_i, K_c], [T_i, 0])  # K_c*(T_i*s+1)/(T_i*s)
print(G_c)

In [ ]:
# Closed-loop simulation: PI regulator + continuous-time plant
# Includes 2nd-order critically-damped measurement filter: F_m(s) = 1/(T_f_m*s+1)^2
# (matches the filter structure inside the reference PIDController)

T_f_m = 0.2  # measurement filter time constant [s]

# 2nd-order critically-damped measurement filter
F_m = con.tf([1], [T_f_m**2, 2 * T_f_m, 1])

# Closed-loop TFs with measurement filter in the feedback path:
#   G_cl   = G_c*G_p / (1 + G_c*F_m*G_p)   (r -> y)
#   G_dist = G_p    / (1 + G_c*F_m*G_p)     (p -> y)
G_cl = con.feedback(G_c * G_p, F_m)
G_dist = con.feedback(G_p, G_c * F_m)
print("Closed-loop poles:", G_cl.poles().round(2))

t = np.linspace(0, 50, 1001)
r = np.where(t >= 1, 1.0, 0.0)
p = np.where(t >= 26, 1.0, 0.0)

response_r = con.forced_response(G_cl, T=t, U=r)
response_p = con.forced_response(G_dist, T=t, U=p)
y = response_r.y[0] + response_p.y[0]

# Control signal: u = G_c * (r - F_m*y)
ym = con.forced_response(F_m, T=t, U=y).y[0]
u = con.forced_response(G_c, T=t, U=r - ym).y[0]

data = pd.DataFrame(
    {"reference": r, "disturbance": p, "input": u, "output": y},
    index=pd.Index(t, name="Time (s)"),
)

plot_info = {
    "Closed-Loop Disturbance Response": {
        "Reference $r(t)$": {
            "var_name": "reference",
            "kind": "step",
            "linestyle": "--",
            "color": "r",
        },
        "Disturbance $p(t)$": {
            "var_name": "disturbance",
            "kind": "step",
            "linestyle": "--",
            "color": "k",
        },
        "Output $y(t)$": {"var_name": "output"},
    },
    "Control Input": {
        "Input $u(t)$": {"var_name": "input", "color": "C1"},
    },
}

fig, axes = make_tsplots(data, plot_info=plot_info)
plt.tight_layout()
plt.show()

In [ ]:
con.pzmap(G_cl)
plt.show()

In [ ]:
G_cl.bode_plot()
plt.tight_layout()
plt.show()

## Compare to Discrete Time Regulator - PI

In [ ]:
from python_pid import PIDController

Ts = 0.2  # sample time [s]
# T_f_m defined above; TfTs = T_f_m / Ts

k_p, k_i, k_d = pid_interactive_ct_to_linear_parallel_dt(
    K_c, T_i, T_d=0.0, Ts=Ts
)
TfTs = T_f_m / Ts

ctrl = PIDController(k_p, k_i, k_d, TfTs=TfTs)
ctrl

In [ ]:
# Discrete-time simulation: discretized G_c vs. reference PIDController
# Both use a 2nd-order critically-damped measurement filter (time constant: T_f_m)
#
# PIDController gains scaled for sample time Ts:
#   ki_dt = ki_ct * Ts  (kd_dt = 0 since T_d = 0)

G_p_d = G_p.sample(Ts, method="zoh")
G_c_d = G_c.sample(Ts, method="tustin")
F_m_d = F_m.sample(Ts, method="tustin")  # same 2nd-order filter, discretized

G_p_ss = con.ss(G_p_d)
G_c_ss = con.ss(G_c_d)
F_m_ss = con.ss(F_m_d)

print("Discrete plant (ZOH):")
print(G_p_d)
print("\nDiscrete PI controller (Tustin):")
print(G_c_d)
print("\nDiscrete measurement filter (Tustin):")
print(F_m_d)

t_end = 50.0
N = int(t_end / Ts) + 1
t_d = np.arange(N) * Ts
r_d = np.where(t_d >= 1.0, 1.0, 0.0)
p_d = np.where(t_d >= 26.0, 1.0, 0.0)

# --- Simulation 1: discretized G_c with 2nd-order measurement filter ---
x_p = np.zeros(G_p_ss.nstates)
x_c = np.zeros(G_c_ss.nstates)
x_fm = np.zeros(F_m_ss.nstates)
y_ctrl = np.zeros(N)
u_ctrl = np.zeros(N)

for k in range(N):
    y_k = float((G_p_ss.C @ x_p).item())
    ym_k = float((F_m_ss.C @ x_fm + F_m_ss.D[0, 0] * y_k).item())
    e_k = r_d[k] - ym_k
    u_k = float((G_c_ss.C @ x_c + G_c_ss.D[0, 0] * e_k).item())
    y_ctrl[k] = y_k
    u_ctrl[k] = u_k
    x_fm = F_m_ss.A @ x_fm + F_m_ss.B.flatten() * y_k
    x_c = G_c_ss.A @ x_c + G_c_ss.B.flatten() * e_k
    x_p = G_p_ss.A @ x_p + G_p_ss.B.flatten() * (u_k + p_d[k])

# --- Simulation 2: reference PIDController (TfTs = T_f_m / Ts) ---
ctrl.reset()
x_p = np.zeros(G_p_ss.nstates)
y_pid = np.zeros(N)
u_pid = np.zeros(N)

for k in range(N):
    y_k = float((G_p_ss.C @ x_p).item())
    u_k = ctrl(r_d[k], y_k, Tx=1.0)
    y_pid[k] = y_k
    u_pid[k] = u_k
    x_p = G_p_ss.A @ x_p + G_p_ss.B.flatten() * (u_k + p_d[k])

In [ ]:
# Compare continuous-time, discretized G_c, and reference PIDController responses
fig, axes = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)

ax1, ax2 = axes

ax1.plot(t, r, "k--", label="Reference $r$")
ax1.plot(t, y, "C0", label="Continuous-time PI")
ax1.step(
    t_d,
    y_ctrl,
    "C1",
    where="post",
    label="Discretized $G_c$ (Tustin, $T_s$=1 s)",
)
ax1.step(t_d, y_pid, "C2", where="post", label="Reference PI controller")
ax1.set_ylabel("Output $y$")
ax1.legend()
ax1.grid(True)

ax2.plot(t, u, "C0", label="Continuous-time PI")
ax2.step(
    t_d,
    u_ctrl,
    "C1",
    where="post",
    label="Discretized $G_c$ (Tustin, $T_s$=1 s)",
)
ax2.step(t_d, u_pid, "C2", where="post", label="Reference PI controller")
ax2.set_ylabel("Control input $u$")
ax2.set_xlabel("Time (s)")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Same comparison using make_tsplots.
# CT and DT data have different time indices so we call make_tsplots twice:
# first for the CT results (creates fig/axes), then for the DT results (reuses axes).

data_ct = pd.DataFrame(
    {"reference": r, "output": y, "input": u},
    index=pd.Index(t, name="Time (s)"),
)
data_dt = pd.DataFrame(
    {
        "output_ctrl": y_ctrl,
        "input_ctrl": u_ctrl,
        "output_pid": y_pid,
        "input_pid": u_pid,
    },
    index=pd.Index(t_d, name="Time (s)"),
)

plot_info_ct = {
    "Closed-Loop Disturbance Response": {
        "Reference $r$": {
            "var_name": "reference",
            "linestyle": "--",
            "color": "k",
            "linewidth": 1,
        },
        "Continuous-time PI": {
            "var_name": "output",
            "color": "C0",
            "linewidth": 1.5,
        },
    },
    "Control Input": {
        "Continuous-time PI": {
            "var_name": "input",
            "color": "C0",
            "linewidth": 1.5,
        },
    },
}
plot_info_dt = {
    "Closed-Loop Disturbance Response": {
        f"Discretized $G_c$": {
            "var_name": "output_ctrl",
            "kind": "step",
            "color": "C1",
            "linewidth": 1.5,
        },
        "Reference PI": {
            "var_name": "output_pid",
            "kind": "step",
            "color": "C2",
            "linewidth": 1.5,
        },
    },
    "Control Input": {
        f"Discretized $G_c$": {
            "var_name": "input_ctrl",
            "kind": "step",
            "color": "C1",
            "linewidth": 1.5,
        },
        "Reference PI": {
            "var_name": "input_pid",
            "kind": "step",
            "color": "C2",
            "linewidth": 1.5,
        },
    },
}

fig, axes = make_tsplots(data_ct, plot_info=plot_info_ct)
fig, axes = make_tsplots(data_dt, plot_info=plot_info_dt, axes=axes)
plt.tight_layout()
plt.show()

## Test Case 2

In [ ]:
# Second-order plant with RHP zero (non-minimum phase)
K_p = 2.0
T_1 = 2.0
T_2 = 3.0
T_0 = -1.0  # negative: RHP zero at s = -1/T_0 = 1

G_p = con.tf([K_p * T_0, K_p], np.convolve([T_1, 1], [T_2, 1]))
print(G_p)

In [ ]:
con.pzmap(G_p)
plt.show()

In [ ]:
# Open-loop step response
t = np.linspace(0, 20, 201)
u_ol = np.where(t >= 1, 1.0, 0.0)

response = con.forced_response(G_p, T=t, U=u_ol)
y_ol = response.y[0]

data = pd.DataFrame(
    {"input": u_ol, "output": y_ol}, index=pd.Index(t, name="Time (t)")
)

plot_info = {
    "Step Response": {
        "Input $u(t)$": {
            "var_name": "input",
            "kind": "step",
            "linestyle": "--",
            "color": "k",
        },
        "Output $y(t)$": {"var_name": "output"},
    }
}

fig, axes = make_tsplots(data, plot_info=plot_info)
plt.tight_layout()
plt.show()

In [ ]:
# PID design rule (simplified lambda / SIMC-style)
K_c = 1.0 / K_p
T_i = T_1
T_d = T_2

if T_0 > 0.0:
    T_f = T_0
else:
    T_f = T_d / 5

K_c, T_i, T_d, T_f

In [ ]:
# PID controller in interactive (series) form
# G_c(s) = K_c*(T_i*s+1)*(T_d*s+1) / (T_i*s*(T_f*s+1))
G_c = con.tf(
    K_c * np.convolve([T_i, 1], [T_d, 1]), np.convolve([T_i, 0], [T_f, 1])
)
print(G_c)

In [ ]:
# Closed-loop simulation: PID regulator + continuous-time plant
# 2-DOF structure: u = G_pi*r - G_c*y
#   - G_c applied to y: derivative filter (T_f*s+1) in G_c denominator acts on measurement
#   - G_pi (PI only) applied to r: no derivative kick on setpoint changes

# PI component for setpoint path (no derivative)
G_pi = con.tf([K_c * T_i, K_c], [T_i, 0])  # K_c*(T_i*s+1) / (T_i*s)

# Closed-loop TFs — loop gain G_c*G_p unchanged
G_dist = con.feedback(G_p, G_c)  # p -> y:  G_p / (1 + G_c*G_p)
G_cl = G_pi * G_dist  # r -> y:  G_pi*G_p / (1 + G_c*G_p)
print("Closed-loop poles:", G_cl.poles().round(2))

t = np.linspace(0, 50, 1001)
r = np.where(t >= 1, 1.0, 0.0)
p = np.where(t >= 26, 1.0, 0.0)

response_r = con.forced_response(G_cl, T=t, U=r)
response_p = con.forced_response(G_dist, T=t, U=p)
y = response_r.y[0] + response_p.y[0]

# Control signal: u = G_pi*r - G_c*y
u_pi = con.forced_response(G_pi, T=t, U=r).y[0]
u_c = con.forced_response(G_c, T=t, U=-y).y[0]
u = u_pi + u_c

data = pd.DataFrame(
    {"reference": r, "disturbance": p, "input": u, "output": y},
    index=pd.Index(t, name="Time (s)"),
)

plot_info = {
    "Closed-Loop Disturbance Response": {
        "Reference $r(t)$": {
            "var_name": "reference",
            "kind": "step",
            "linestyle": "--",
            "color": "r",
        },
        "Disturbance $p(t)$": {
            "var_name": "disturbance",
            "kind": "step",
            "linestyle": "--",
            "color": "k",
        },
        "Output $y(t)$": {"var_name": "output"},
    },
    "Control Input": {
        "Input $u(t)$": {"var_name": "input", "color": "C1"},
    },
}

fig, axes = make_tsplots(data, plot_info=plot_info)
plt.tight_layout()
plt.show()

In [ ]:
con.pzmap(G_cl)
plt.show()

In [ ]:
G_cl.bode_plot()
plt.tight_layout()
plt.show()

## Compare to Discrete Time Regulator - PID

In [ ]:
Ts = 0.2  # sample time [s]

k_p, k_i, k_d = pid_interactive_ct_to_linear_parallel_dt(K_c, T_i, T_d, Ts=Ts)

# Use a smaller value for TfTs so it is closer to the 1st order filter T_f
# Set it to half of T_f/Ts to approximate the 2nd-order filter with the 1st-order T_f   
TfTs = T_f / Ts / 2

# To bridge the 2-DOF interactive form and the parallel incremental form we need to 
# adjust the setpoint weight, b.
# With b=1.0 (default), the reference PIDController applies kp = K_c*(T_i+T_d)/T_i
# to the setpoint error, while the 2-DOF G_c structure only applies K_c to the setpoint 
# (via G_pi), leaving the portion of kp that comes from the derivative term acting on
# the measurement only. This causes a mismatch in the initial control action at t=1.0 
# when r steps from 0 to 1.
# E.g.
# At t=1.0, r steps to 1 and y=0. The PIDController gives u=1.30:
#   Dup = kp*(b*r - yf) - up_old = 1.25*(1.0 * 1 - 0) - 0 = 1.25
#   Dui = ki*(r - yf)*Tx         = 0.05 * 1 * 1         = 0.05
#   u   = 0 + 1.25 + 0.05 = 1.30

# The discretized G_c gives u=0.525 because it uses G_pi (PI only) on r, not the full
# PID kp:
#   u_pi = G_pi_ss.D * r = 0.525 * 1 = 0.525   (kp_pi = K_c = 0.5)
#   u_c  = G_c_ss.D * (-y) = 0                  (y=0 at this step)

# The root cause is setpoint weighting b. The PIDController defaults to b=1, so
# the full kp = K_c*(T_i+T_d)/T_i = 1.25 acts on r. But the 2-DOF G_c structure
# only applies the PI gain K_c = 0.5 to the setpoint — the extra derivative 
# contribution to kp only acts on the measurement.

# The two are equivalent when:
# $$b = \frac{k_{p,\text{PI}}}{k_{p,\text{PID}}} 
#     = \frac{K_c}{K_c(T_i+T_d)/T_i}
#     = \frac{T_i}{T_i + T_d}
#     = \frac{2}{2+3} = 0.4$$

# Conclusion:
#   set b = T_i / (T_i + T_d) in the PIDController call to match the 2-DOF interactive form.
b = T_i / (T_i + T_d)

ctrl = PIDController(k_p, k_i, k_d, b=b, TfTs=TfTs)
ctrl

In [ ]:
# Discrete-time simulation: discretized python-control G_c vs. reference PIDController
#
# 2-DOF structure: u = G_pi*r - G_c*y
#   - G_c on -y: internal (T_f*s+1) filter acts on measurement (derivative-on-measurement)
#   - G_pi on r: no derivative kick on setpoint changes
#
# PIDController (TfTs = T_f/Ts): 2nd-order filter vs G_c's 1st-order (T_f*s+1)

# Discretize plant (ZOH), setpoint path and full controller (Tustin)
G_p_d = G_p.sample(Ts, method="zoh")
G_pi_d = G_pi.sample(Ts, method="tustin")
G_c_d = G_c.sample(Ts, method="tustin")

G_p_ss = con.ss(G_p_d)
G_pi_ss = con.ss(G_pi_d)
G_c_ss = con.ss(G_c_d)

print("Discrete plant (ZOH):")
print(G_p_d)
print("\nDiscrete PI setpoint path (Tustin):")
print(G_pi_d)
print("\nDiscrete controller G_c (Tustin):")
print(G_c_d)

t_end = 50.0
N = int(t_end / Ts) + 1
t_d = np.arange(N) * Ts
r_d = np.where(t_d >= 1.0, 1.0, 0.0)
p_d = np.where(t_d >= 26.0, 1.0, 0.0)

# --- Simulation 1: discretized 2-DOF controller (G_pi on r, G_c on -y) ---
x_p = np.zeros(G_p_ss.nstates)
x_pi = np.zeros(G_pi_ss.nstates)
x_c = np.zeros(G_c_ss.nstates)
y_ctrl = np.zeros(N)
u_ctrl = np.zeros(N)

for k in range(N):
    y_k = float((G_p_ss.C @ x_p).item())
    u_pi_k = float((G_pi_ss.C @ x_pi + G_pi_ss.D[0, 0] * r_d[k]).item())
    u_c_k = float((G_c_ss.C @ x_c + G_c_ss.D[0, 0] * (-y_k)).item())
    u_k = u_pi_k + u_c_k
    y_ctrl[k] = y_k
    u_ctrl[k] = u_k
    x_pi = G_pi_ss.A @ x_pi + G_pi_ss.B.flatten() * r_d[k]
    x_c = G_c_ss.A @ x_c + G_c_ss.B.flatten() * (-y_k)
    x_p = G_p_ss.A @ x_p + G_p_ss.B.flatten() * (u_k + p_d[k])

# --- Simulation 2: reference PIDController (2nd-order filter) ---
ctrl.reset()
x_p = np.zeros(G_p_ss.nstates)
y_pid = np.zeros(N)
u_pid = np.zeros(N)

for k in range(N):
    y_k = float((G_p_ss.C @ x_p).item())
    u_k = ctrl(r_d[k], y_k, Tx=1.0)
    y_pid[k] = y_k
    u_pid[k] = u_k
    x_p = G_p_ss.A @ x_p + G_p_ss.B.flatten() * (u_k + p_d[k])

In [ ]:
# Compare continuous-time, discretized G_c, and reference PIDController responses
fig, axes = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)

ax1, ax2 = axes

ax1.plot(t, r, "k--", label="Reference $r$")
ax1.plot(t, y, "C0", label="Continuous-time PID")
ax1.step(
    t_d,
    y_ctrl,
    "C1",
    where="post",
    label=f"Discretized $G_c$ (Tustin, $T_s$={Ts} s)",
)
ax1.step(t_d, y_pid, "C2", where="post", label="Reference PID controller")
ax1.set_ylabel("Output $y$")
ax1.legend()
ax1.grid(True)

ax2.plot(t, u, "C0", label="Continuous-time PID")
ax2.step(
    t_d,
    u_ctrl,
    "C1",
    where="post",
    label=f"Discretized $G_c$ (Tustin, $T_s$={Ts} s)",
)
ax2.step(t_d, u_pid, "C2", where="post", label="Reference PID controller")
ax2.set_ylabel("Control input $u$")
ax2.set_xlabel("Time (s)")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Same comparison using make_tsplots.
data_ct = pd.DataFrame(
    {"reference": r, "output": y, "input": u},
    index=pd.Index(t, name="Time (s)"),
)
data_dt = pd.DataFrame(
    {
        "output_ctrl": y_ctrl,
        "input_ctrl": u_ctrl,
        "output_pid": y_pid,
        "input_pid": u_pid,
    },
    index=pd.Index(t_d, name="Time (s)"),
)

plot_info_ct = {
    "Closed-Loop Disturbance Response": {
        "Reference $r$": {
            "var_name": "reference",
            "linestyle": "--",
            "color": "k",
            "linewidth": 1,
        },
        "Continuous-time PID": {
            "var_name": "output",
            "color": "C0",
            "linewidth": 1.5,
        },
    },
    "Control Input": {
        "Continuous-time PID": {
            "var_name": "input",
            "color": "C0",
            "linewidth": 1.5,
        },
    },
}
plot_info_dt = {
    "Closed-Loop Disturbance Response": {
        f"Discretized $G_c$": {
            "var_name": "output_ctrl",
            "kind": "step",
            "color": "C1",
            "linewidth": 1.5,
        },
        "Reference PID": {
            "var_name": "output_pid",
            "kind": "step",
            "color": "C2",
            "linewidth": 1.5,
        },
    },
    "Control Input": {
        f"Discretized $G_c$": {
            "var_name": "input_ctrl",
            "kind": "step",
            "color": "C1",
            "linewidth": 1.5,
        },
        "Reference PID": {
            "var_name": "input_pid",
            "kind": "step",
            "color": "C2",
            "linewidth": 1.5,
        },
    },
}

fig, axes = make_tsplots(data_ct, plot_info=plot_info_ct)
fig, axes = make_tsplots(data_dt, plot_info=plot_info_dt, axes=axes)
plt.tight_layout()
plt.show()

In [ ]:
print(data_ct.iloc[15:30])

In [ ]:
print(data_dt.iloc[:20])